# 07 — Citation Analysis and Related Work Selection

Identifica trabajos influyentes, recientes, revisiones y candidatos para cada subsección de Related Work.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar corpus y secciones

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

df = load_clean_corpus(PROJECT_ROOT)
sections = read_json(PROJECT_ROOT / "config" / "related_work_sections.json")


## 2. Indicadores de citación

In [ ]:

df["Cited by numeric"] = pd.to_numeric(df["Cited by"], errors="coerce").fillna(0)
df["Year numeric"] = pd.to_numeric(df["Year"], errors="coerce")
search_year = int(CONFIG["search_date"][:4])
df["Citation age"] = (search_year - df["Year numeric"] + 1).clip(lower=1)
df["Citations per year"] = (df["Cited by numeric"] / df["Citation age"]).round(3)
top_cited = df.sort_values(["Cited by numeric", "Citations per year"], ascending=[False, False])
top_recent_impact = df.sort_values(["Citations per year", "Cited by numeric"], ascending=[False, False])
reviews = df[df["Document Type"].str.lower().eq("review")].sort_values("Cited by numeric", ascending=False)
save_csv(top_cited[[c for c in ["Title", "Year", "Authors", "Source title normalized", "Cited by", "Citations per year", "DOI"] if c in top_cited.columns]], OUTPUT_DIR / "07_top_cited_documents.csv")
save_csv(top_recent_impact[[c for c in ["Title", "Year", "Authors", "Source title normalized", "Cited by", "Citations per year", "DOI"] if c in top_recent_impact.columns]], OUTPUT_DIR / "07_top_citations_per_year.csv")
save_csv(reviews[[c for c in ["Title", "Year", "Authors", "Source title normalized", "Cited by", "DOI"] if c in reviews.columns]], OUTPUT_DIR / "07_review_articles.csv")
display(top_cited[["Title", "Year", "Cited by numeric", "Citations per year"]].head(30))


## 3. Selección reproducible de Related Work

In [ ]:

text = combined_text(df)
frames = []
for section, patterns in sections.items():
    pattern = "|".join("(?:{})".format(item) for item in patterns)
    mask = text.str.contains(pattern, regex=True, na=False)
    candidates = df.loc[mask].copy()
    candidate_text = text.loc[mask]
    candidates["Match count"] = candidate_text.str.count(pattern)
    candidates["Citation signal"] = (candidates["Cited by numeric"] + 1) ** 0.5
    valid_years = candidates["Year numeric"].dropna()
    min_year = valid_years.min() if not valid_years.empty else search_year
    max_year = valid_years.max() if not valid_years.empty else search_year
    candidates["Recency signal"] = (candidates["Year numeric"].fillna(min_year) - min_year + 1) / max(1, max_year - min_year + 1)
    candidates["Review bonus"] = candidates["Document Type"].str.lower().eq("review").astype(int) * 1.5
    candidates["Selection score"] = candidates["Match count"] * 3 + candidates["Citation signal"] + candidates["Recency signal"] + candidates["Review bonus"]
    candidates["Related Work section"] = section
    candidates = candidates.sort_values(["Selection score", "Cited by numeric", "Year numeric"], ascending=[False, False, False])
    frames.append(candidates.head(int(CONFIG.get("max_related_work_candidates_per_section", 40))))
related = pd.concat(frames, ignore_index=True)
columns = [c for c in ["Related Work section", "Title", "Year", "Authors", "Source title normalized", "Document Type", "Cited by", "Citations per year", "DOI", "Match count", "Selection score", "Abstract"] if c in related.columns]
save_csv(related[columns], OUTPUT_DIR / "07_related_work_candidates.csv")
display(related[columns].groupby("Related Work section").head(10))


## 4. Documentos más cercanos al gap

In [ ]:

theme_docs = pd.read_csv(OUTPUT_DIR / "04_theme_document_matches.csv", encoding="utf-8-sig")
gap_docs = theme_docs[theme_docs["Family"] == "Semantic/Assistance"].copy()
gap_docs["Cited by numeric"] = pd.to_numeric(gap_docs["Cited by"], errors="coerce").fillna(0)
gap_docs = gap_docs.sort_values(["Concept", "Cited by numeric", "Year"], ascending=[True, False, False]).drop_duplicates(["EID", "Concept"])
save_csv(gap_docs, OUTPUT_DIR / "07_semantic_gap_documents.csv")
display(gap_docs.head(50))


## 5. Nota metodológica

In [ ]:

text_note = (
    "The automated ranking is a discovery aid. Candidate papers were scored using thematic matches, citation signal, recency, "
    "and a modest review-article bonus. Every selected paper must be read before inclusion in the manuscript."
)
(REPORTS_DIR / "07_related_work_methodology.txt").write_text(text_note, encoding="utf-8")
print(text_note)
